In [7]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import IsolationForest
import warnings

warnings.filterwarnings('ignore')

print("=" * 80)
print("Step 4: CF 품질 평가 및 최적 CF 선별 (Best-of-N Selection)")
print("=" * 80)

# ============================================================================
# 1. 데이터 로드
# ============================================================================
print("\n[Step 1] 데이터 로드")
print("-" * 80)

# CF 생성 결과 로드
try:
    cf_results = pd.read_csv('cf_results_full.csv')
    print(f"CF 데이터 로드 완료: {cf_results.shape}")
except FileNotFoundError:
    raise FileNotFoundError("'cf_results_full.csv' 파일이 없습니다. Step 3를 먼저 실행하세요.")

# 모델 및 변수명 로드
selected_features = joblib.load('selected_features_final.pkl')
model = joblib.load('base_model_final.pkl')
threshold = joblib.load('base_model_threshold_final.pkl')

# Realism 계산을 위한 원본 학습 데이터 로드 (분포 확인용)
# Step 1에서 저장한 원본 데이터를 사용하는 것이 가장 정확함
df_train = pd.read_csv('selected_data_for_modeling.csv')
X_train = df_train[selected_features].values

print(f"분석 대상 기업 수: {cf_results['ID'].nunique()}개")
print(f"평가할 CF 총 개수: {len(cf_results)}개")

# ============================================================================
# 2. 데이터 정규화 (MinMaxScaler) - 거리 계산용
# ============================================================================
print("\n[Step 2] 데이터 정규화 (거리 계산용)")

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train) # 학습 데이터 기준으로 스케일러 학습

# ============================================================================
# 3. 개별 CF 품질 평가 함수 정의
# ============================================================================
def evaluate_single_cf(row, scaler, model, threshold, X_train_scaled, iso_forest):
    """
    단일 CF에 대한 5가지 품질 메트릭 계산
    """
    # 1. 데이터 준비
    original_vals = np.array([row[f'Original_{feat}'] for feat in selected_features])
    cf_vals = np.array([row[f'CF_{feat}'] for feat in selected_features])
    
    # 스케일링 (거리 계산용)
    orig_scaled = scaler.transform(original_vals.reshape(1, -1))[0]
    cf_scaled = scaler.transform(cf_vals.reshape(1, -1))[0]
    
    # 2. 메트릭 계산
    
    # (1) Validity: 목표 확률 달성 여부 (이미 생성 단계에서 확인했으나 재검증)
    # 1.0 = 성공, 0.0 = 실패 (확률값 자체를 점수에 반영할 수도 있음)
    pred_prob = model.predict_proba(cf_vals.reshape(1, -1))[0, 1]
    validity = 1.0 if pred_prob < threshold else 0.0
    
    # (2) Proximity: 원본과의 거리 (L1 Distance) - 낮을수록 좋음
    # 정규화된 공간에서의 거리
    proximity = np.mean(np.abs(orig_scaled - cf_scaled))
    
    # (3) Sparsity: 변하지 않은 변수의 비율 - 높을수록 좋음
    # 변화량이 매우 작은(1e-6 이하) 경우 변경 안 된 것으로 간주
    changes = np.abs(orig_scaled - cf_scaled)
    sparsity = np.mean(changes < 1e-6)
    
    # (4) Realism: 데이터 분포 적합도 (Isolation Forest) - 높을수록 좋음
    iso_score = iso_forest.decision_function(cf_scaled.reshape(1, -1))[0]
    # 점수 정규화 (대략 0~1 사이로 매핑, 시각적 편의 위함)
    realism = (iso_score + 0.5) if iso_score > -0.5 else 0.0 # 단순화된 매핑
    
    # (5) Robustness: 노이즈에 대한 강건성 - 높을수록 좋음
    n_perturb = 10
    noise_level = 0.01
    stable_cnt = 0
    for _ in range(n_perturb):
        noise = np.random.normal(0, noise_level, size=cf_vals.shape)
        # 스케일에 맞춰 노이즈 조정 필요하나 단순화
        perturbed_cf = cf_vals + (noise * (cf_vals + 1e-6)) 
        p_prob = model.predict_proba(perturbed_cf.reshape(1, -1))[0, 1]
        if p_prob < threshold:
            stable_cnt += 1
    robustness = stable_cnt / n_perturb
    
    return {
        'Validity': validity,
        'Proximity': proximity, # 낮을수록 좋음
        'Sparsity': sparsity,   # 높을수록 좋음
        'Realism': realism,     # 높을수록 좋음
        'Robustness': robustness # 높을수록 좋음
    }

# ============================================================================
# 4. 품질 평가 실행
# ============================================================================
print("\n[Step 3] 전체 CF 품질 평가 수행")
print("※ Isolation Forest 학습 및 전체 데이터 평가 중...")

# Isolation Forest 학습 (Realism 평가용)
iso_forest = IsolationForest(n_estimators=100, contamination='auto', random_state=42, n_jobs=-1)
iso_forest.fit(X_train_scaled)

evaluated_cfs = []

# tqdm이 있다면 사용, 없으면 그냥 진행
try:
    from tqdm import tqdm
    iterator = tqdm(cf_results.iterrows(), total=len(cf_results), desc="Evaluating")
except ImportError:
    iterator = cf_results.iterrows()

for idx, row in iterator:
    metrics = evaluate_single_cf(row, scaler, model, threshold, X_train_scaled, iso_forest)
    
    # 결과 저장
    res = row.to_dict()
    res.update(metrics)
    
    # 종합 점수 (Quality Score) 계산
    # 수식: (Validity * 10) + (1-Proximity) + Sparsity + Realism + Robustness
    # Validity가 0이면 점수가 크게 깎이도록 설계 (Hard Constraint 효과)
    score = (
        (metrics['Validity'] * 5.0) +  # Validity가 제일 중요 (가중치 5배)
        (1.0 - metrics['Proximity']) + # 가까울수록 점수 Up
        metrics['Sparsity'] +          # 적게 변할수록 점수 Up
        metrics['Realism'] +           # 현실적일수록 점수 Up
        metrics['Robustness']          # 튼튼할수록 점수 Up
    ) / 9.0 # 정규화 (대략 0~1 범위)
    
    res['Quality_Score'] = score
    evaluated_cfs.append(res)

df_evaluated = pd.DataFrame(evaluated_cfs)

print(f"\n평가 완료: 총 {len(df_evaluated)}개 CF")

# ============================================================================
# 5. 최적 CF 선별 (Best-of-N)
# ============================================================================
print("\n[Step 4] 기업별 최적 CF 선별 (Top 1 Selection)")

# 기업 ID별로 그룹화하여 Quality Score가 가장 높은 행만 추출
# idxmax()를 사용하여 각 그룹에서 최대 스코어를 가진 인덱스를 찾음
best_indices = df_evaluated.groupby('ID')['Quality_Score'].idxmax()
df_best = df_evaluated.loc[best_indices].copy()

# Validity가 0인(실패한) 케이스 걸러내기
# 모델 예측상 여전히 '부도'라면 의미가 없으므로 제외
df_final = df_best[df_best['Validity'] == 1.0].copy()

print(f"선별 전 기업 수: {cf_results['ID'].nunique()}")
print(f"선별 후 기업 수: {len(df_final)} (Validity 실패 제외)")
print(f"탈락된 기업 수: {cf_results['ID'].nunique() - len(df_final)}")

# ============================================================================
# 6. 결과 저장
# ============================================================================
print("\n[Step 5] 결과 저장")

# 최종 선별된 데이터 저장 (Agent가 사용할 파일)
output_file = 'cf_results_filtered.csv'
df_final.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"최적 CF 저장 완료: {output_file}")

# 전체 평가 내역 저장 (참고용)
df_evaluated.to_csv('cf_evaluation_details.csv', index=False, encoding='utf-8-sig')
print(f"전체 평가 내역 저장 완료: cf_evaluation_details.csv")

# ============================================================================
# 7. 샘플 확인
# ============================================================================
if not df_final.empty:
    print("\n[최적 CF 샘플 확인 (Top 1)]")
    sample = df_final.iloc[0]
    print(f"ID: {sample['ID']}")
    print(f"Quality Score: {sample['Quality_Score']:.4f}")
    print(f"선택된 CF 번호: #{sample['CF_Number']}")
    print(f"부도 확률 변화: {sample['Original_Proba']:.4f} -> {sample['Target_Proba']:.4f}")
    
    print("\n[품질 메트릭]")
    print(f"- Validity:   {sample['Validity']:.1f}")
    print(f"- Proximity:  {sample['Proximity']:.4f} (낮을수록 좋음)")
    print(f"- Sparsity:   {sample['Sparsity']:.4f} (높을수록 좋음)")
    print(f"- Realism:    {sample['Realism']:.4f}")
    print(f"- Robustness: {sample['Robustness']:.4f}")
    
    print("\n[주요 변화 내용]")
    for feat in selected_features:
        change = sample[f'Change_{feat}']
        if abs(change) > 0.001: # 변화가 있는 변수만 출력
            print(f"- {feat}: {sample[f'Original_{feat}']} -> {sample[f'CF_{feat}']} (Delta: {change:+.4f})")
else:
    print("\n[Warning] 유효한 CF가 하나도 없습니다.")

print("\nStep 4 완료. 이제 Agent #2가 이 파일을 사용하여 보고서를 작성합니다.")

Step 4: CF 품질 평가 및 최적 CF 선별 (Best-of-N Selection)

[Step 1] 데이터 로드
--------------------------------------------------------------------------------
CF 데이터 로드 완료: (1061, 196)
분석 대상 기업 수: 266개
평가할 CF 총 개수: 1061개

[Step 2] 데이터 정규화 (거리 계산용)

[Step 3] 전체 CF 품질 평가 수행
※ Isolation Forest 학습 및 전체 데이터 평가 중...


Evaluating: 100%|█████████████████████████████████████████████████████████████████| 1061/1061 [00:10<00:00, 102.85it/s]



평가 완료: 총 1061개 CF

[Step 4] 기업별 최적 CF 선별 (Top 1 Selection)
선별 전 기업 수: 266
선별 후 기업 수: 266 (Validity 실패 제외)
탈락된 기업 수: 0

[Step 5] 결과 저장
최적 CF 저장 완료: cf_results_filtered.csv
전체 평가 내역 저장 완료: cf_evaluation_details.csv

[최적 CF 샘플 확인 (Top 1)]
ID: 75.0
Quality Score: 0.9074
선택된 CF 번호: #1.0
부도 확률 변화: 0.9899 -> 0.4692

[품질 메트릭]
- Validity:   1.0
- Proximity:  0.0010 (낮을수록 좋음)
- Sparsity:   0.4844 (높을수록 좋음)
- Realism:    0.6831
- Robustness: 1.0000

[주요 변화 내용]
- FN1_1: 646202.0 -> 368768.0 (Delta: -277434.0000)
- FN1_2: 154149.0 -> 80552.0 (Delta: -73597.0000)
- FN1_3: 644576.0 -> 233569.0 (Delta: -411007.0000)
- FN1_4: 1626.0 -> 135199.0 (Delta: +133573.0000)
- FN1_5: 79466.0 -> 65921.0 (Delta: -13545.0000)
- FN1_11: 469183.0 -> 0.0 (Delta: -469183.0000)
- FN1_13: 800351.0 -> 449320.0 (Delta: -351031.0000)
- FN1_14: 499354.0 -> 41452.0 (Delta: -457902.0000)
- FN1_16: 17346.0 -> 136166.0 (Delta: +118820.0000)
- FN1_17: 486255.0 -> 0.0 (Delta: -486255.0000)
- FN1_18: 17346.0 -> 136166.0 (Delta: +

### CF 성능 메트릭 통계량 비교(전체 vs Selected)

In [1]:
import pandas as pd
import numpy as np

print("=" * 80)
print("논문용 성능 메트릭 비교표 생성 (Raw vs Selected)")
print("=" * 80)

# 1. 데이터 로드 (Step 4에서 생성된 파일)
try:
    # 모든 CF(4개씩)의 평가 결과가 담긴 파일
    df_all = pd.read_csv('cf_evaluation_details.csv')
    print(f"전체 데이터 로드: {len(df_all)}건")
except FileNotFoundError:
    print("오류: 'cf_evaluation_details.csv' 파일이 없습니다. Step 4를 확인하세요.")

# 2. 그룹 나누기
# Group A: 전체 생성된 CF (Raw Generated)
# Group B: 각 기업별로 Quality Score가 가장 높은 1개 (Selected Best)
# (이미 Step 4 로직에 따라 Validity=1.0인 것 중 1등만 뽑아야 함)

# Validity가 1.0(성공)인 것 중에서만 선별하는 것이 공정한 비교일 수 있으나,
# '전체 생성 성능'을 보여주려면 Validity 실패 포함 전체를 평균 내는 것도 의미 있음.
# 여기서는 논문 논리를 위해 '전체(Raw)' vs '최종 채택(Selected)'로 비교합니다.

# 최적 CF 선별 (Step 4와 동일 로직)
best_indices = df_all[df_all['Validity'] == 1.0].groupby('ID')['Quality_Score'].idxmax()
df_selected = df_all.loc[best_indices]

print(f"선별된 데이터: {len(df_selected)}건")

# 3. 메트릭 비교 계산
metrics = ['Validity', 'Proximity', 'Sparsity', 'Realism', 'Robustness', 'Quality_Score']
results = []

for metric in metrics:
    # 전체 평균 (Raw)
    mean_all = df_all[metric].mean()
    std_all = df_all[metric].std()
    
    # 선별 평균 (Selected)
    mean_sel = df_selected[metric].mean()
    std_sel = df_selected[metric].std()
    
    # 개선율 (Improvement)
    # Proximity는 낮을수록 좋으므로 계산 반대
    if metric == 'Proximity':
        imp = (mean_all - mean_sel) / mean_all * 100 # 감소해야 좋음 (+)
    else:
        imp = (mean_sel - mean_all) / mean_all * 100 # 증가해야 좋음 (+)
        
    results.append({
        'Metric': metric,
        'Raw (Mean ± Std)': f"{mean_all:.3f} ± {std_all:.3f}",
        'Selected (Mean ± Std)': f"{mean_sel:.3f} ± {std_sel:.3f}",
        'Improvement (%)': f"{imp:+.1f}%"
    })

# 4. 결과 출력 및 저장
df_comparison = pd.DataFrame(results)
print("\n[논문용 메트릭 비교표]")
print(df_comparison.to_markdown(index=False)) # 마크다운 형태로 출력

# CSV 저장
df_comparison.to_csv('thesis_metric_comparison.csv', index=False, encoding='utf-8-sig')
print("\n저장 완료: thesis_metric_comparison.csv")

논문용 성능 메트릭 비교표 생성 (Raw vs Selected)
전체 데이터 로드: 1061건
선별된 데이터: 266건

[논문용 메트릭 비교표]
| Metric        | Raw (Mean ± Std)   | Selected (Mean ± Std)   | Improvement (%)   |
|:--------------|:-------------------|:------------------------|:------------------|
| Validity      | 1.000 ± 0.000      | 1.000 ± 0.000           | +0.0%             |
| Proximity     | 0.004 ± 0.009      | 0.001 ± 0.002           | +67.7%            |
| Sparsity      | 0.475 ± 0.081      | 0.501 ± 0.129           | +5.6%             |
| Realism       | 0.668 ± 0.025      | 0.676 ± 0.013           | +1.2%             |
| Robustness    | 1.000 ± 0.000      | 1.000 ± 0.000           | +0.0%             |
| Quality_Score | 0.904 ± 0.009      | 0.908 ± 0.014           | +0.5%             |

저장 완료: thesis_metric_comparison.csv
